In [1]:
!python -m pip install numpy pandas librosa tensorflow


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!python -m pip install tensorflow==2.20.0


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import argparse
import json
import os
import re
import numpy as np
import pandas as pd
import librosa

In [1]:
import tensorflow as tf

: 

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
!git add .
!git commit -m "Update code"
!git push --force-with-lease origin main

In [1]:
!git push --force origin main

Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 8 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (10/10), 10.28 KiB | 10.28 MiB/s, done.
Total 10 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Varshh-hub/DL-END-TO-END.git
 + 9b44b61...2d76012 main -> main (forced update)


In [ ]:
SAMPLE_RATE = 16000
N_MFCC = 40
MAX_SECONDS = 12
MAX_FRAMES = int(MAX_SECONDS * 100) + 2 

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z' ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
def extract_features(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=MAX_SECONDS)
    if len(y) < int(0.15 * SAMPLE_RATE):
        return None
    mfcc = librosa.feature.mfcc(
        y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
        n_fft=400, hop_length=160
    ).T.astype("float32")
    # Per-utterance normalization
    mfcc = (mfcc - mfcc.mean(axis=0, keepdims=True)) / (mfcc.std(axis=0, keepdims=True) + 1e-5)
    return mfcc[:MAX_FRAMES]

In [ ]:
def make_model(vocab_size):
    audio = layers.Input(shape=(None, N_MFCC), dtype="float32", name="audio")
    labels = layers.Input(shape=(None,), dtype="int32", name="labels")
    input_len = layers.Input(shape=(1,), dtype="int32", name="input_length")
    label_len = layers.Input(shape=(1,), dtype="int32", name="label_length")

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(audio)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    probs = layers.Dense(vocab_size + 1, activation="softmax", name="character_probabilities")(x)

    def ctc_loss(args):
        y_true, y_pred, ilen, llen = args
        return tf.keras.backend.ctc_batch_cost(y_true, y_pred, ilen, llen)

    loss = layers.Lambda(ctc_loss, name="ctc_loss")([labels, probs, input_len, label_len])
    training_model = Model([audio, labels, input_len, label_len], loss)
    training_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                           loss=lambda _y, y_pred: y_pred)
    inference_model = Model(audio, probs)
    return training_model, inference_model

In [ ]:
def batch_generator(records, char_to_id, batch_size, shuffle=True):
    while True:
        order = np.random.permutation(len(records)) if shuffle else np.arange(len(records))
        for start in range(0, len(order), batch_size):
            batch_records = [records[i] for i in order[start:start+batch_size]]
            feats, labs = [], []
            for path, text in batch_records:
                f = extract_features(path)
                if f is None or not text:
                    continue
                ids = [char_to_id[c] for c in text if c in char_to_id]
                # CTC needs enough frames to place the characters.
                if len(ids) == 0 or len(f) < len(ids):
                    continue
                feats.append(f)
                labs.append(ids)
            if not feats:
                continue
            max_t = max(len(x) for x in feats)
            max_l = max(len(x) for x in labs)
            x = np.zeros((len(feats), max_t, N_MFCC), dtype="float32")
            for i, f in enumerate(feats):
                x[i, :len(f)] = f
            y = pad_sequences(labs, maxlen=max_l, padding="post", value=0, dtype="int32")
            ilen = np.array([[len(f)] for f in feats], dtype="int32")
            llen = np.array([[len(s)] for s in labs], dtype="int32")
            inputs = {"audio": x, "labels": y, "input_length": ilen, "label_length": llen}
            # The model's output is already the loss.
            yield inputs, np.zeros((len(feats), 1), dtype="float32")

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--tsv", required=True, help="Common Voice validated.tsv (or similar TSV)")
    parser.add_argument("--clips", required=True, help="Directory containing audio clips")
    parser.add_argument("--limit", type=int, default=3000)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=8)
    args = parser.parse_args()

    df = pd.read_csv(args.tsv, sep="\t")
    if not {"path", "sentence"}.issubset(df.columns):
        raise ValueError("TSV must contain 'path' and 'sentence' columns.")
    records = []
    for row in df[["path", "sentence"]].dropna().itertuples(index=False):
        text = clean_text(row.sentence)
        path = os.path.join(args.clips, row.path)
        if text and os.path.isfile(path):
            records.append((path, text))
        if len(records) >= args.limit:
            break
    if len(records) < 100:
        raise ValueError(f"Only found {len(records)} usable clips. Check --tsv and --clips paths.")

    # Character 0 is reserved for padding; CTC blank is the final output class.
    chars = sorted(set("".join(text for _, text in records)))
    char_to_id = {c: i for i, c in enumerate(chars)}
    os.makedirs("artifacts", exist_ok=True)
    with open("artifacts/vocab.json", "w", encoding="utf-8") as f:
        json.dump({"char_to_id": char_to_id, "id_to_char": {str(v): k for k, v in char_to_id.items()}}, f, ensure_ascii=False, indent=2)

    train_model, inference_model = make_model(len(chars))
    steps = max(1, int(np.ceil(len(records) / args.batch_size)))
    train_model.fit(
        batch_generator(records, char_to_id, args.batch_size),
        steps_per_epoch=steps,
        epochs=args.epochs,
        verbose=1
    )
    inference_model.save("artifacts/speech_model.keras")
    print("Saved artifacts/speech_model.keras and artifacts/vocab.json")

if __name__ == "__main__":
    main()